## Mini Project — Exploratory Data Analysis on India's COVID-19 Data

# 🇮🇳 COVID-19 in India: State-wise Trends, Recovery & Vaccination Analysis

## 📌 Project Objective

This notebook analyses the COVID-19 pandemic in India using **NumPy, Pandas, Matplotlib, and Seaborn**.

The analysis is divided into two connected phases:

1. **First Wave (30 Jan 2020 – 6 Aug 2020)**
   - State-wise confirmed cases
   - Recoveries
   - Deaths
   - Recovery and mortality rates
   - State-wise trends

2. **Vaccination Rollout (15 Jan 2021 onward)**
   - Total vaccinations
   - Daily vaccination progress
   - Vaccination milestones
   - Cumulative vaccination growth

### 🎯 Key Questions

- Which states reported the highest number of COVID-19 cases?
- Which states had the highest recovery rates?
- How did deaths vary across states?
- Which states experienced the fastest growth?
- How did India's vaccination programme progress over time?
- What major trends can be observed from the first wave and subsequent vaccination rollout?

> **Important:** The case dataset ends on 6 August 2020, while vaccination data starts from January 2021. Therefore, vaccination data is analysed as a separate subsequent phase rather than directly comparing vaccination with first-wave cases.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

## Data

The case dataset contains daily state-wise COVID-19 information for India.

Source:
`imdevskp/covid-19-india-data`

Coverage:
**30 January 2020 – 6 August 2020**

In [ ]:
cases = pd.read_csv(
    "https://raw.githubusercontent.com/imdevskp/covid-19-india-data/master/complete.csv"
)
vax = pd.read_csv(
    "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/country_data/India.csv"
)

print("Cases shape:", cases.shape)
print("Vaccination shape:", vax.shape)
cases.head()

## 3. Initial Data Overview

In [ ]:
print("Shape:", cases.shape)

In [ ]:
# Check data types and non-null values
cases.info()

In [ ]:
# Statistical summary of numerical columns
cases.describe(include="all")

In [ ]:
# Missing values in each column
cases.isnull().sum()

In [ ]:
# Display all column names
cases.columns.tolist()

## 4. Data Cleaning

In [ ]:
# Rename columns with simple and consistent names
cases.columns = [
    "date", "state", "lat", "long", "confirmed",
    "deaths", "cured", "new_cases", "new_deaths", "new_recovered"
]

# Display the new column names
cases.columns

In [ ]:
# Convert the date column into datetime format
cases["date"] = pd.to_datetime(cases["date"])

# Check the data type
cases["date"].dtype

In [ ]:
# Convert deaths into numeric values
# Invalid values are converted to NaN
cases["deaths"] = pd.to_numeric(
    cases["deaths"],
    errors="coerce"
)

# Replace missing values with 0
cases["deaths"] = cases["deaths"].fillna(0)

# Convert to integer
cases["deaths"] = cases["deaths"].astype(int)

In [ ]:
# Convert confirmed and recovered cases into integers
cases["confirmed"] = cases["confirmed"].astype(int)
cases["cured"] = cases["cured"].astype(int)

In [ ]:
# Check the number of missing values
cases.isnull().sum()

In [ ]:
# Count duplicate rows
print("Duplicate rows:", cases.duplicated().sum())

In [ ]:
# Create a dictionary to correct state names
name_fix = {
    "Telangana***": "Telangana",
    "Telengana": "Telangana",
    "Union Territory of Jammu and Kashmir": "Jammu and Kashmir",
    "Union Territory of Ladakh": "Ladakh",
    "Union Territory of Chandigarh": "Chandigarh"
}

# Replace inconsistent names
cases["state"] = cases["state"].replace(name_fix)

In [ ]:
# Remove duplicate records for the same date and state
cases = cases.drop_duplicates(
    subset=["date", "state"]
)

In [ ]:
# Sort the data by state and date
cases = cases.sort_values(
    ["state", "date"]
).reset_index(drop=True)

In [ ]:
# Calculate active COVID-19 cases
# Active = Confirmed - Deaths - Recovered

cases["active"] = (
    cases["confirmed"]
    - cases["deaths"]
    - cases["cured"]
)

cases.head()

In [ ]:
# Check the cleaned dataset
print("Dataset Shape:", cases.shape)

# Check unique state names
print("\nNumber of States/UTs:", cases["state"].nunique())

# Check data types
print("\nData Types:")
print(cases.dtypes)

# Check missing values again
print("\nMissing Values:")
print(cases.isnull().sum())

## 5. National-Level Analysis

In [ ]:
# Group all states by date
# Then calculate the total confirmed cases, deaths and recoveries
national = cases.groupby(
    "date",
    as_index=False
)[["confirmed", "deaths", "cured"]].sum()

# Display the first few rows
national.head()

In [ ]:
# Recovery Rate = (Recovered / Confirmed) × 100
national["recovery_rate"] = (
    national["cured"] / national["confirmed"] * 100
).round(2)

national.head()

In [ ]:
# Death Rate = (Deaths / Confirmed) × 100
national["death_rate"] = (
    national["deaths"] / national["confirmed"] * 100
).round(2)

national.head()

In [ ]:
# Calculate the number of new confirmed cases each day
national["new_confirmed"] = national["confirmed"].diff()

# For the first day, use the total confirmed cases
national["new_confirmed"] = national["new_confirmed"].fillna(
    national["confirmed"]
)

national.head()

In [ ]:
# Find the latest date in the dataset
latest_date = cases["date"].max()

# Get the latest national statistics
latest_row = national.iloc[-1]

# Display the latest COVID-19 statistics
print(f"COVID-19 Statistics as of {latest_date.date()}:")
print(f"Total Confirmed : {latest_row['confirmed']:,}")
print(f"Total Recovered : {latest_row['cured']:,}")
print(f"Total Deaths    : {latest_row['deaths']:,}")
print(f"Recovery Rate   : {latest_row['recovery_rate']}%")
print(f"Death Rate      : {latest_row['death_rate']}%")

In [ ]:
# Make sure the national data is arranged chronologically
national = national.sort_values("date").reset_index(drop=True)

national.head()

## 6. State-wise Analysis

In [ ]:
# Find the latest date in the dataset
latest_date = cases["date"].max()

# Select data for the latest date
latest = cases[cases["date"] == latest_date].copy()

# Display the latest state-wise data
latest.head()

In [ ]:
# Calculate recovery rate for each state
latest["recovery_rate"] = (
    latest["cured"] / latest["confirmed"] * 100
).round(2)

In [ ]:
# Calculate death rate for each state
latest["death_rate"] = (
    latest["deaths"] / latest["confirmed"] * 100
).round(2)

In [ ]:
# Sort states by confirmed cases
latest = latest.sort_values(
    "confirmed",
    ascending=False
).reset_index(drop=True)

# Select the top 10 states
top10 = latest.head(10)

# Display important columns
top10[
    [
        "state",
        "confirmed",
        "deaths",
        "cured",
        "recovery_rate",
        "death_rate"
    ]
]

## 7. Monthly State-wise Analysis

In [ ]:
# Extract year and month from the date
cases["month"] = cases["date"].dt.strftime("%Y-%m")

# Check the new column
cases[["date", "month"]].head()

In [ ]:
# Select the top 5 states
top5_states = top10["state"].head(5)

# Filter data for these states
top5_data = cases[cases["state"].isin(top5_states)]

# Create monthly pivot table
month_pivot = top5_data.pivot_table(
    values="new_cases",
    index="state",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

# Display the pivot table
month_pivot

# 8. Visualisations (12 charts)

# Chart 1 -> National cumulative trend

In [ ]:
# Create the figure
plt.figure(figsize=(10, 5))

# Plot confirmed cases
plt.plot(
    national["date"],
    national["confirmed"],
    label="Confirmed"
)

# Plot recovered cases
plt.plot(
    national["date"],
    national["cured"],
    label="Recovered"
)

# Plot deaths
plt.plot(
    national["date"],
    national["deaths"],
    label="Deaths"
)

# Add title and labels
plt.title("India: Cumulative COVID-19 Cases Over Time")
plt.xlabel("Date")
plt.ylabel("Number of People")

# Show legend
plt.legend()

# Rotate date labels
plt.xticks(rotation=45)

# Adjust the layout
plt.tight_layout()

# Display the chart
plt.show()

# Chart 2 -> Daily New Confirmed Cases

In [ ]:
# Create the figure
plt.figure(figsize=(10, 5))

# Create a bar chart
plt.bar(
    national["date"],
    national["new_confirmed"]
)

# Add title and labels
plt.title("India: Daily New Confirmed Cases")
plt.xlabel("Date")
plt.ylabel("New Cases")

# Rotate date labels for better readability
plt.xticks(rotation=45)

# Adjust the layout
plt.tight_layout()

# Display the chart
plt.show()

# Chart 3 -> Top 10 States by Total Confirmed Cases

In [ ]:
# Create the figure
plt.figure(figsize=(10, 5))

# Create a horizontal bar chart
plt.barh(
    top10["state"],
    top10["confirmed"]
)

plt.title(
    f"Top 10 States by Total Confirmed Cases (as of {latest_date.date()})"
)
plt.xlabel("Confirmed Cases")
plt.ylabel("State")

plt.tight_layout()
plt.show()

# Chart 4 -> Recovery Rate of the Top 10 Worst-Hit States

In [ ]:
# Sort states by recovery rate
top10_sorted = top10.sort_values("recovery_rate")

# Create the figure
plt.figure(figsize=(10, 5))

# Create a horizontal bar chart
plt.barh(
    top10_sorted["state"],
    top10_sorted["recovery_rate"]
)

plt.title("Recovery Rate (%) — Top 10 Worst-Hit States")
plt.xlabel("Recovery Rate (%)")
plt.ylabel("State")

plt.tight_layout()
plt.show()

# Chart 5 -> Confirmed-Case Trend for the Top 5 States

In [ ]:
# Get the names of the top 5 states
top5_states = top10["state"].head(5).tolist()

# Create the figure
plt.figure(figsize=(10, 5))

# Plot confirmed cases for each state
for state in top5_states:
    state_data = cases[cases["state"] == state]

    plt.plot(
        state_data["date"],
        state_data["confirmed"],
        label=state
    )

plt.title("Confirmed Cases Over Time — Top 5 States")
plt.xlabel("Date")
plt.ylabel("Confirmed Cases")
plt.legend(fontsize=8)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Chart 6 -> Share of National Cases by State

In [ ]:
# Get the top 6 states by confirmed cases
top6 = latest.nlargest(6, "confirmed")[["state", "confirmed"]]

# Calculate cases from all other states
others = latest["confirmed"].sum() - top6["confirmed"].sum()

# Create labels
pie_labels = top6["state"].tolist() + ["Rest of India"]

# Create values
pie_values = top6["confirmed"].tolist() + [others]

# Create the figure
plt.figure(figsize=(7, 7))

# Create the pie chart
plt.pie(
    pie_values,
    labels=pie_labels,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Share of Total Confirmed Cases by State")

plt.tight_layout()
plt.show()

# Chart 7 -> National Recovery-Rate Trend

In [ ]:
# Create the figure
plt.figure(figsize=(10, 5))

# Plot the recovery rate
plt.plot(
    national["date"],
    national["recovery_rate"],
    linewidth=2
)

plt.title("India: National Recovery Rate Over Time")
plt.xlabel("Date")
plt.ylabel("Recovery Rate (%)")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Chart 8 -> Recovery Rate vs Death Rate by State

In [ ]:
# Select states with at least 1,000 confirmed cases
sizeable = latest[latest["confirmed"] >= 1000]

# Create the figure
plt.figure(figsize=(8, 6))

# Create the scatter plot
plt.scatter(
    sizeable["recovery_rate"],
    sizeable["death_rate"],
    s=sizeable["confirmed"] / 20,
    alpha=0.6
)

# Label the top 6 states by confirmed cases
for _, row in sizeable.nlargest(6, "confirmed").iterrows():
    plt.annotate(
        row["state"],
        (row["recovery_rate"], row["death_rate"]),
        fontsize=8,
        xytext=(3, 3),
        textcoords="offset points"
    )

plt.title("Recovery Rate vs Death Rate by State")
plt.xlabel("Recovery Rate (%)")
plt.ylabel("Death Rate (%)")

plt.tight_layout()
plt.show()

# Chart 9 -> Correlation Heatmap of National Metrics

In [ ]:
# Select the metrics for correlation analysis
corr_cols = [
    "confirmed",
    "deaths",
    "cured",
    "new_confirmed",
    "recovery_rate",
    "death_rate"
]

# Calculate the correlation matrix
corr = national[corr_cols].corr()

# Display the heatmap
plt.figure(figsize=(8, 6))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between National COVID Metrics")

plt.tight_layout()
plt.show()

# Chart 10 -> Spread of Daily New Cases by Month

In [ ]:
# Create a month column
national["month"] = national["date"].dt.strftime("%Y-%m")

# Create the figure
plt.figure(figsize=(10, 5))

# Create the boxplot
sns.boxplot(
    data=national,
    x="month",
    y="new_confirmed"
)

plt.title("Spread of Daily New Cases by Month")
plt.xlabel("Month")
plt.ylabel("Daily New Cases")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Chart 11 -> India's Vaccination Progress (2021–2024)

In [ ]:
# Convert date column to datetime
vax["date"] = pd.to_datetime(vax["date"])

# Create the figure
plt.figure(figsize=(10, 5))

# Plot total vaccinations in billions
plt.plot(
    vax["date"],
    vax["total_vaccinations"] / 1e9,
    label="Total Doses (Bn)",
    linewidth=2
)

# Plot fully vaccinated people in billions
plt.plot(
    vax["date"],
    vax["people_fully_vaccinated"] / 1e9,
    label="Fully Vaccinated (Bn)",
    linewidth=2
)

plt.title("India: COVID-19 Vaccination Progress (2021–2024)")
plt.xlabel("Date")
plt.ylabel("People / Doses (in Billions)")
plt.legend()

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Chart 12 -> The 10 Least-Affected States/UTs

In [ ]:
# Remove states with zero confirmed cases
positive_cases = latest[latest["confirmed"] > 0]

# Select the 10 states/UTs with the lowest confirmed cases
bottom10 = positive_cases.nsmallest(10, "confirmed")

# Create the figure
plt.figure(figsize=(10, 5))

# Create a horizontal bar chart
plt.barh(
    bottom10["state"],
    bottom10["confirmed"]
)

plt.title(
    f"10 Least-Affected States/UTs by Confirmed Cases "
    f"(as of {latest_date.date()})"
)
plt.xlabel("Confirmed Cases")
plt.ylabel("State / UT")

plt.tight_layout()
plt.show()

# 9. Save the cleaned dataset

In [ ]:
# Save the cleaned data as a CSV file
cases.to_csv("cleaned_covid_india.csv",index=False)

# Display confirmation
print("Saved cleaned_covid_india.csv —",cases.shape[0],"rows.")